# Importo librerie necessarie


In [15]:
from re import match

from pymongo import MongoClient
from scripts import methods
from bson.objectid import ObjectId
from pandasql import sqldf
import pandas as pd

# Attivazione del Client MongoDB

In [16]:
# Chiamo il client
client = MongoClient("mongodb://localhost:27017/")

#Acquisisco il DB
db = client["Keyblade"]

#Prendo in esempio il dataset videogames_2016
videogames_2016 = db["videogames_2016"]
videogames_2024 = db["videogames_2024"]

# Query per la visualizzazione di giochi per rating (2016)

In [17]:
#Visualizzo i rating distinti presenti nel dataset videogames_2016
ratings=videogames_2016.distinct("Rating")

#creo un dizionario per la lista dei nomi per ciascun rating
rating_list={}

# Per ogni rating eseguo aggregazione
for rating in ratings:
    pipeline = [
        {"$match": {"Rating": rating}},
        {"$project": {"Name": 1, "Platform": 1, "_id": 0}}
    ]
    games = videogames_2016.aggregate(pipeline)
    games_df = pd.DataFrame(list(games))
    rating_list[rating] = games_df

# Visualizzazione dei risultati
for rating, games_df in rating_list.items():
    print(f"Giochi con rating {rating}:")
    display(games_df)
    print("\n")


# Query per la visualizzazione delle vendite totali dei giochi in base al Developer (Video giochi 2016)

In [19]:
# Recupero dei developer distinti
developers = videogames_2016.distinct("Developer")

# Lista per raccogliere i risultati
rows = []

# Ciclo sui developer
for developer in developers:
    if developer == "Unknown":
        # Visualizza giochi singoli per Unknown
        pipeline = [
            {"$match": {"Developer": "Unknown"}},
            {"$project": {"Developer": 1,"Global_Sales": 1, "_id": 0}}
        ]
        result = list(videogames_2016.aggregate(pipeline))
        rows.extend(result)
    else:
        # Somma le vendite per gli altri developer
        pipeline = [
            {"$match": {"Developer": developer}},
            {"$group": {
                "_id": "$Developer",
                "Global_Sales": {"$sum": "$Global_Sales"}
            }}
        ]
        result = list(videogames_2016.aggregate(pipeline))
        if result:
            rows.append({"Developer": developer, "Global_Sales": result[0]["Global_Sales"]})
        else:
            rows.append({"Developer": developer,"Global_Sales": 0})

# Creazione del DataFrame
sales_df = pd.DataFrame(rows)
# Ordinamento per Global_Sales
sales_df = sales_df.sort_values(by="Global_Sales", ascending=False)

# Visualizzazione
display(sales_df)



# Query per la visualizzazione delle vendite totali dei giochi in base al Developer (Video giochi 2024)

In [21]:
# Recupero dei developer distinti
developers = videogames_2024.distinct("developer")

# Lista per raccogliere i risultati
rows = []

# Ciclo sui developer
for developer in developers:
    if developer == "Unknown":
        # Visualizza giochi singoli per Unknown
        pipeline = [
            {"$match": {"developer": "Unknown"}},
            {"$project": {"developer": 1, "total_sales": 1, "_id": 0}}
        ]
        result = list(videogames_2024.aggregate(pipeline))
        rows.extend(result)
    else:
        # Somma le vendite per gli altri developer
        pipeline = [
            {"$match": {"developer": developer}},
            {"$group": {
                "_id": "$developer",
                "total_sales": {"$sum": "$total_sales"}
            }}
        ]
        result = list(videogames_2024.aggregate(pipeline))
        if result:
            rows.append({"developer": developer, "total_sales": result[0]["total_sales"]})
        else:
            rows.append({"developer": developer, "total_sales": 0})

# Creazione del DataFrame
sales_df = pd.DataFrame(rows)
# Ordinamento per total_sales
sales_df = sales_df.sort_values(by="total_sales", ascending=False)

# Visualizzazione
display(sales_df)


# Visualizzazione con dettagli di vendita (2016)

In [23]:
# Pipeline: visualizza tutti i giochi con dettagli di vendita
pipeline = [
    {"$project": {
        "_id": 0,
        "Developer": 1,
        "Global_Sales": 1,
        "NA_Sales": 1,
        "EU_Sales": 1,
        "JP_Sales": 1,
        "Other_Sales": 1
    }},
    {"$sort": {"Global_Sales": -1}}
]#Inserendo il -1 nell'operazione di sort, i risultati vengono ordinati in ordine decrescente in base alle vendite globali

# Esecuzione della pipeline
result = list(videogames_2016.aggregate(pipeline))
# Conversione in DataFrame
sales_df = pd.DataFrame(result)

# Visualizzazione
display(sales_df)


# Giochi pubblicati in un determinato range temporale  (tramite publisher - 2024)

In [31]:
pipeline = [
        {
            "$match": {
                "publisher": {
                    "$regex": "Rockstar",  # Match publisher name or part of it
                    "$options": "i"  # Case-insensitive match
                },
            "release_date": {
                "$gte" : 2010
            }
            }
        },
        {
            "$project": {
                "_id": 0,
                "title": 1,
                "console": 1,
                "release_date": 1,
                "genre": 1,
                "publisher": 1,
                "total_sales": 1
            }
        }
    ]

publisher_df = pd.DataFrame(rows)
result = list(videogames_2024.aggregate(pipeline))
publisher_df = pd.DataFrame(result)
display(publisher_df)

,title,console,genre,publisher,total_sales,release_date
0,Grand Theft Auto V,PS3,Action,Rockstar Games,20.32,2013
1,Grand Theft Auto V,PS4,Action,Rockstar Games,19.39,2014
2,Grand Theft Auto V,X360,Action,Rockstar Games,15.86,2013
3,Red Dead Redemption 2,PS4,Action-Adventure,Rockstar Games,13.94,2018
4,Grand Theft Auto V,XOne,Action,Rockstar Games,8.72,2014
...,...,...,...,...,...,...
63,The Warriors,PS3,Misc,Rockstar Games,0.00,2013
64,Midnight Club: Los Angeles Complete Edition,PSN,Racing,Rockstar Games,0.00,2011
65,Max Payne 3,All,Shooter,Rockstar Games,0.00,2012
66,Max Payne 3,PC,Shooter,Rockstar Games,0.00,2012


# Ottiene le vendite in base al genere (2024)

In [32]:
pipeline = [
        {"$match": {"genre": "Action"}},  # Match games with genre "Action"
        {"$project": {
            "_id": 0,
            "title": 1,
            "console": 1,
            "release_date": 1,
            "genre": 1,
            "publisher": 1,
            "total_sales": 1
        }},
        {"$sort": {"total_sales": -1}} # Sort by Global_Sales in descending order
    ]

genre_df = pd.DataFrame(rows)
result = list(videogames_2024.aggregate(pipeline))
genre_df = pd.DataFrame(result)
display(genre_df)

,title,console,genre,publisher,total_sales,release_date
0,Grand Theft Auto V,PS3,Action,Rockstar Games,20.32,2013
1,Grand Theft Auto V,PS4,Action,Rockstar Games,19.39,2014
2,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,16.15,2002
3,Grand Theft Auto V,X360,Action,Rockstar Games,15.86,2013
4,Grand Theft Auto III,PS2,Action,Rockstar Games,13.10,2001
...,...,...,...,...,...,...
8549,Zoids Wild: Blast Unleashed,NS,Action,Outright Games,0.00,2020
8550,Zoids Wild: King of Blast,NS,Action,Takara Tomy,0.00,2019
8551,Zombeer,WiiU,Action,Unknown,0.00,Unknown
8552,Zombie Brigade: No Brain No Gain,WiiU,Action,Unknown,0.00,Unknown
